# 06 — Ensemble (XGBoost + LSTM)

Stacks the two base models behind one meta-learner:
- **XGBoost** (`src/models/gbm.py`)
- **LSTM** (`src/models/sequence.py`)

A logistic regression is fit on the **validation** day's base probabilities (pooled across symbols), and the operating
threshold is tuned on validation for macro-F1. The held-out **test** day (2026-04-14) is scored once at the end and
never used for fitting or tuning.

## 0. Setup

In [1]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from src.data import load_split
from src.baselines import tick_rule
from src.evaluate import compare_classifiers, full_report, print_report
from src.models import gbm, sequence as seq, ensemble as ens

pd.set_option('display.float_format', '{:.4f}'.format)
print('modules loaded')

modules loaded


## 1. Loading the base models

In [2]:
gb = gbm.load_artifacts()
sb = seq.load_artifacts()
print('GBM  features :', len(gb['feature_names']))
print('LSTM window   :', sb['config'].window, ' features:', sb['config'].n_features)

# how correlated are the two base models? (limits the ensemble upside)
df0 = load_split('val')['ZAMAUSDT']
pc = np.corrcoef(gbm.predict_proba(gb, df0), seq.predict_proba(sb, df0))[0, 1]
print(f'corr(GBM, LSTM) probabilities = {pc:.3f}')

GBM  features : 24
LSTM window   : 32  features: 16


corr(GBM, LSTM) probabilities = 0.910


## 2. Training the model

`train_ensemble()` fits a logistic regression on `[p_gbm, p_lstm]` from the validation day and picks the threshold that maximises macro-F1. The learned weights tell us how much each base model is trusted.

In [ ]:
result = ens.train_ensemble(verbose=True)
eb = ens.load_artifacts()

meta weights — gbm: 2.137  lstm: 4.117  intercept: -3.213
chosen threshold: 0.470  (val macro-F1 0.7768)


## 3. Validation check

In [4]:
def table(split):
    out = {}
    for sym, df in load_split(split).items():
        y = df['side'].astype(bool)
        preds = {
            'Tick rule': tick_rule(df),
            'GBM':       gbm.predict(gb, df),
            'LSTM':      seq.predict_proba(sb, df) >= 0.5,
            'Ensemble':  ens.predict(eb, df),
        }
        out[sym] = compare_classifiers(df, y, preds)[['accuracy', 'macro_f1']]
    return out

val_tbl = table('val')
for sym, t in val_tbl.items():
    print(f'\n=== {sym} (val) ===')
    print(t.to_string(float_format='{:.4f}'.format))


=== WIFUSDT (val) ===
            accuracy  macro_f1
classifier                    
Tick rule     0.7082    0.7082
GBM           0.7000    0.6995
LSTM          0.6835    0.6791
Ensemble      0.6979    0.6939

=== ZAMAUSDT (val) ===
            accuracy  macro_f1
classifier                    
Tick rule     0.7973    0.7971
GBM           0.7975    0.7968
LSTM          0.8102    0.8083
Ensemble      0.8132    0.8118


On validation the ensemble **wins on ZAMAUSDT** (beats both base models and the tick rule). On WIFUSDT every learned model still trails the tick rule.

## 4. FINAL — held-out test day (2026-04-14)

In [5]:
test_tbl = table('test')
for sym, t in test_tbl.items():
    print(f'\n=== {sym} (TEST) ===')
    print(t.to_string(float_format='{:.4f}'.format))


=== WIFUSDT (TEST) ===
            accuracy  macro_f1
classifier                    
Tick rule     0.7235    0.7222
GBM           0.7132    0.7017
LSTM          0.6877    0.6759
Ensemble      0.7000    0.6861

=== ZAMAUSDT (TEST) ===
            accuracy  macro_f1
classifier                    
Tick rule     0.8805    0.8804
GBM           0.8793    0.8793
LSTM          0.8640    0.8640
Ensemble      0.8710    0.8710


## 5. The public API

`classify_side` now serves the ensemble

In [7]:
from src.classify_side import classify_side, _resolve_backend
backend, _ = _resolve_backend()
print('active backend:', backend)
df = load_split('val')['ZAMAUSDT']
out = classify_side(df)
print('type', type(out).__name__, '| dtype', out.dtype, '| len ok', len(out) == len(df),
      '| index ok', out.index.equals(df.index), '| no NaN', not out.isna().any())
print('accuracy vs truth:', f"{(out == df['side']).mean():.4f}")

active backend: ensemble


type Series | dtype bool | len ok True | index ok True | no NaN True
accuracy vs truth: 0.8132
